# 02 · As três configurações: sem contexto · RAG · agente

Cobre os itens **04** (geração fundamentada: JSON válido, citação e recusa) e
**05** (especialização por **agente** comparada ao RAG simples) da pontuação.

> ⚠️ Requer a LLM local no ar: `llama-server` rodando na porta 8080 (ver README).

**Por que agente em vez de LoRA?** O fine-tuning LoRA exige GPU NVIDIA/CUDA ou treino em nuvem.
A especialização por **agente** roda 100% local na GPU AMD (RX 6600, via Vulkan) e ataca
diretamente os erros observados no RAG simples: JSON inválido, citação errada e alucinação.
O agente adiciona 4 etapas: **buscar com reformulação → gerar → validar JSON/citação → autocrítica**.

In [1]:
import sys, json
sys.path.append("..")
from src.busca import Busca
from src.configuracoes import sem_contexto, rag, agente
from src import llm

print("LLM local disponível?", llm.disponivel())
busca = Busca()

LLM local disponível? True


## Pergunta respondível pela base

In [2]:
pergunta = "Caí num buraco com o carro. A prefeitura de São Paulo tem que pagar o conserto?"

for fn in (sem_contexto, rag, agente):
    r = fn(pergunta, busca)
    print(f"\n===== {r['config']} =====")
    print(json.dumps(r["json"], ensure_ascii=False, indent=2))
    for p in r["passos"]:
        print("  ·", p)


===== sem_contexto =====
{
  "resposta": "não consta",
  "fonte_id": null,
  "confianca": 0.0
}
  · pergunta direta ao modelo, sem contexto

===== rag =====
{
  "resposta": "não consta",
  "fonte_id": "ctb-02",
  "confianca": 0.0
}
  · busca top-3: jur-01=0.20, ind-03=0.13, sp156-01=0.12
  · resposta gerada com os trechos no prompt

===== agente =====
{
  "resposta": "não consta",
  "fonte_id": "ctb-02",
  "confianca": 0.0
}
  · busca 1: jur-01=0.20, ind-03=0.13, sp156-01=0.12
  · resposta gerada
  · modelo recusou com os trechos em mãos


## Pergunta que a base NÃO cobre (esperado: recusa "não consta")

É aqui que o modelo **sem contexto** costuma alucinar — inventar números, leis e telefones —
enquanto o RAG e o agente recusam.

In [3]:
pergunta_nc = "Quantos buracos foram tapados na Avenida Paulista em 2025?"

for fn in (sem_contexto, rag, agente):
    r = fn(pergunta_nc, busca)
    print(f"\n===== {r['config']} =====")
    print(json.dumps(r["json"], ensure_ascii=False, indent=2))
    for p in r["passos"]:
        print("  ·", p)


===== sem_contexto =====
{
  "resposta": "não consta",
  "fonte_id": null,
  "confianca": 0.0
}
  · pergunta direta ao modelo, sem contexto

===== rag =====
{
  "resposta": "não consta",
  "fonte_id": null,
  "confianca": 0.0
}
  · busca top-3: calc-04=0.09, ilum-01=0.08, calc-03=0.08
  · score máximo abaixo do limiar (0.15): recusa

===== agente =====
{
  "resposta": "não consta",
  "fonte_id": null,
  "confianca": 0.0
}
  · busca 1: calc-04=0.09, ilum-01=0.08, calc-03=0.08
  · score baixo; consulta reformulada: 'Quantos consertos de tapa-buracos foram realizados na Avenida Paulista em 2025?'
  · busca 2: sp156-02=0.09, sp156-03=0.07, calc-04=0.07
  · ainda abaixo do limiar (0.15): recusa


## Demonstração de alucinação (para o relatório)

Rode a célula abaixo e guarde o print: o modelo sem contexto respondendo com segurança
algo que não tem como saber.

In [4]:
for p in ["Qual o valor médio das indenizações pagas pela prefeitura de SP?",
          "Qual o telefone direto do subprefeito da Sé?"]:
    r = sem_contexto(p, busca)
    print(f"\nPergunta: {p}")
    print("Saída sem contexto:", json.dumps(r["json"], ensure_ascii=False))


Pergunta: Qual o valor médio das indenizações pagas pela prefeitura de SP?
Saída sem contexto: {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}

Pergunta: Qual o telefone direto do subprefeito da Sé?
Saída sem contexto: {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}


## Triagem de relato (segunda tarefa da IA, usada pelo MVP)

In [5]:
from src.api import TRIAGEM_SISTEMA
relato = "tem um buracão enorme na faixa de ônibus da Av. Santo Amaro, quase caí de moto hoje cedo"
saida = llm.conversar([
    {"role": "system", "content": TRIAGEM_SISTEMA},
    {"role": "user", "content": relato},
])
print(json.dumps(llm.extrair_json(saida), ensure_ascii=False, indent=2))

{
  "tipo": "buraco_corredor_onibus",
  "gravidade": "alta",
  "descricao_curta": "Buraco grande na faixa de ônibus",
  "orgao_sugerido": "SPTrans",
  "local_citado": "Av. Santo Amaro"
}
